## Split data into Train Validation and Test csv's

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Grab file
file_path = "../../ais_data/mid_datav3.csv"
init_df = pd.read_csv(file_path)

In [3]:
print(init_df.dtypes)

Unnamed: 0               int64
MSG_TYPE                 int64
MMSI                     int64
NAME                       str
IMO_NUMBER             float64
CALL_SIGN                  str
LAT_AVG                float64
LON_AVG                float64
PERIOD                     str
SPEED_KNOTS            float64
COG_DEG                float64
HEADING_DEG            float64
NAV_STATUS             float64
NAV_SENSOR             float64
SHIP_AND_CARGO_TYPE      int64
DRAUGHT                float64
DRAUGHT.1              float64
DIM_BOW                  int64
DIM_STERN                int64
DIM_PORT                 int64
DIM_STARBOARD            int64
DESTINATION                str
MMSI_COUNTRY_CD            str
RECEIVER                   str
BEAM                     int64
LENGTH                   int64
time_diff              float64
new_voyage                bool
voyage_id                  str
num_pings                int64
dtype: object


In [4]:
# Only take columns needed for TCN Neural Network
df = init_df[['MMSI', 'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS', 'COG_DEG', 'HEADING_DEG', 'voyage_id']]

# Make a datetime type column
df['TIME'] = pd.to_datetime(df['PERIOD'])
df = df.rename(columns = {'LAT_AVG':'LAT', 'LON_AVG':'LON', 'SPEED_KNOTS':'SPEED', 'COG_DEG':'COG', 'HEADING_DEG':'HEADING'})

print("datetime check", df['TIME'].dtypes)

# drop all NA's
print("pre-NA drop", len(df))
df = df.dropna()
print("post-NA drop", len(df))

datetime check datetime64[us]
pre-NA drop 67363
post-NA drop 64652


In [5]:
# make delta time, lat, and lon columns for TCN
df['dt'] = df['TIME'].diff().dt.total_seconds()
df['dlat'] = df['LAT'].diff()
df['dlon'] = df['LON'].diff()

# sort values by vessel and time
df = df.sort_values(["MMSI", "TIME"]).reset_index(drop=True)

# fast way to change each new voyage delta time, lat, and lon from NaN to 0
mask = df['voyage_id'] != df['voyage_id'].shift()
df.loc[mask, ['dt', 'dlat', 'dlon']] = 0

# older slower way
'''for voyage in range(0, len(df)):
    if voyage > 0 and df.loc[voyage-1, "dt"] != df.loc[voyage, "dt"]:
        df.loc[voyage, ["dt", "dlat", "dlon"]] = 0'''

# change cog and heading to cos and sin for TCN
df['cog_rad'] = np.deg2rad(df['COG'])
df['cog_sin'] = np.sin(df['cog_rad'])
df['cog_cos'] = np.cos(df['cog_rad'])

df['hdg_rad'] = np.deg2rad(df['HEADING'])
df['hdg_sin'] = np.sin(df['hdg_rad'])
df['hdg_cos'] = np.cos(df['hdg_rad'])



df.head()

,MMSI,LAT,LON,PERIOD,SPEED,COG,HEADING,voyage_id,TIME,dt,dlat,dlon,cog_rad,cog_sin,cog_cos,hdg_rad,hdg_sin,hdg_cos
0,205042000,22.736758,-78.616947,2023-06-04 14:00:00,13.0,110.0,111.0,1_205042000,2023-06-04 14:00:00,0.0,0.000000,0.000000,1.919862,0.939693,-0.342020,1.937315,0.933580,-0.358368
1,205042000,22.707576,-78.536280,2023-06-04 14:20:00,13.1,111.2,112.0,1_205042000,2023-06-04 14:20:00,1200.0,-0.029182,0.080667,1.940806,0.932324,-0.361625,1.954769,0.927184,-0.374607
2,205042000,22.705818,-78.531350,2023-06-04 14:25:00,13.1,111.3,112.0,1_205042000,2023-06-04 14:25:00,300.0,-0.001758,0.004930,1.942551,0.931691,-0.363251,1.954769,0.927184,-0.374607
3,205042000,22.666790,-78.419349,2023-06-04 14:55:00,13.3,110.4,111.0,1_205042000,2023-06-04 14:55:00,1800.0,-0.039028,0.112001,1.926843,0.937282,-0.348572,1.937315,0.933580,-0.358368
4,205042000,22.654898,-78.384704,2023-06-04 15:05:00,13.4,110.2,111.0,1_205042000,2023-06-04 15:05:00,600.0,-0.011892,0.034645,1.923353,0.938493,-0.345298,1.937315,0.933580,-0.358368


In [6]:
# Reduce columns again
training_df = df[["MMSI", "voyage_id", "dt", "cog_sin", "cog_cos", "hdg_sin", "hdg_cos", "dlat", "dlon"]]

training_df.head()

,MMSI,voyage_id,dt,cog_sin,cog_cos,hdg_sin,hdg_cos,dlat,dlon
0,205042000,1_205042000,0.0,0.939693,-0.342020,0.933580,-0.358368,0.000000,0.000000
1,205042000,1_205042000,1200.0,0.932324,-0.361625,0.927184,-0.374607,-0.029182,0.080667
2,205042000,1_205042000,300.0,0.931691,-0.363251,0.927184,-0.374607,-0.001758,0.004930
3,205042000,1_205042000,1800.0,0.937282,-0.348572,0.933580,-0.358368,-0.039028,0.112001
4,205042000,1_205042000,600.0,0.938493,-0.345298,0.933580,-0.358368,-0.011892,0.034645


In [7]:
# List of vessels
vessels = df['MMSI'].unique()

# Shuffle for randomness (VERY important)
rng = np.random.default_rng(42)
rng.shuffle(vessels)

n_total = len(vessels)

# ~90% train and ~10% testing
test_split = int(n_total * 0.9)
train_val_vessels = vessels[:test_split]
test_vessels      = vessels[test_split:]

# Split train into ~75% train and ~15% val
val_frac_of_train = 0.2   # ~20% of the ~90%
val_split = int(len(train_val_vessels) * (1 - val_frac_of_train))

train_vessels = train_val_vessels[:val_split]
val_vessels   = train_val_vessels[val_split:]

# Create DataFrames
df_train = df[df['MMSI'].isin(train_vessels)].copy()
df_val   = df[df['MMSI'].isin(val_vessels)].copy()
df_test  = df[df['MMSI'].isin(test_vessels)].copy()

# Diagnostics
print("Train %:", len(df_train)/len(df))
print("Val %:",   len(df_val)/len(df))
print("Test %:",  len(df_test)/len(df))

print("Train rows:", len(df_train))
print("Val rows:",   len(df_val))
print("Test rows:",  len(df_test))

Train %: 0.7187248654333973
Val %: 0.18618140196745653
Test %: 0.09509373259914619
Train rows: 46467
Val rows: 12037
Test rows: 6148


In [8]:
# More Diagnostics
print("number of Train vessels:", df_train["MMSI"].nunique())
print("number of Val vessels:", df_val["MMSI"].nunique())
print("number of Test vessels:", df_test["MMSI"].nunique())

print("number of Train voyages:", df_train["voyage_id"].nunique())
print("number of Val voyages:", df_val["voyage_id"].nunique())
print("number of Test voyages:", df_test["voyage_id"].nunique())

number of Train vessels: 1223
number of Val vessels: 306
number of Test vessels: 170
number of Train voyages: 4253
number of Val voyages: 1042
number of Test voyages: 536


In [9]:
# Create csv's
df_train.to_csv("mid_train_data.csv", index=False)
df_val.to_csv("mid_val_data.csv", index=False)
df_test.to_csv("mid_test_data.csv", index=False)